# CFReT Morphological Signature Significance Plots

This notebook visualizes the statistical significance and effect size of morphological features obtained from the CFReT dataset. By plotting feature statistics—specifically Kolmogorov-Smirnov (KS) effect size and FDR-corrected p-values—we assess which morphological features significantly distinguish failing from healthy cardiac fibroblasts. We split these visualizations into two panels: one highlighting the specific fluorescent channels, and the other dividing features into "on-morphological signatures" and "off-morphological signatures."

In [1]:
suppressPackageStartupMessages({library(arrow)
library(dplyr)
library(ggplot2)
library(tidyr)
library(viridis)
library(RColorBrewer)
library(patchwork) 
library(ggside)
library(IRdisplay)})

Setting input and output paths

In [2]:
# setting signature stats paths
signatures_stats_path <- file.path("../results/signatures/signature_importance.csv")
shuffle_signatures_stats_path <- file.path("../results/signatures/shuffle_signature_importance.csv")

if (!file.exists(signatures_stats_path)) {
  stop(paste("File not found:", signatures_stats_path))
}
if (!file.exists(shuffle_signatures_stats_path)) {
  stop(paste("File not found:", shuffle_signatures_stats_path))
}

# setting output path for the generated plot
sig_plot_output_dir = file.path("./figures")
if (!dir.exists(sig_plot_output_dir)) {
  dir.create(sig_plot_output_dir, showWarnings = FALSE, recursive = TRUE)
}

## Load the Signature Statistics Data

We load the `signature_importance.csv` file, which contains the metrics required to evaluate feature significance.

In [3]:
# Load both signature stats files and label each by shuffled status
sig_stats_df <- read.csv(signatures_stats_path)
sig_stats_df$data_type <- "Non-shuffled"

shuffle_stats_df <- read.csv(shuffle_signatures_stats_path)
shuffle_stats_df$data_type <- "Shuffled"

# Combine into a single dataframe and set factor order so non-shuffled appears on top
combined_df <- rbind(sig_stats_df, shuffle_stats_df)
combined_df$data_type <- factor(combined_df$data_type, levels = c("Non-shuffled", "Shuffled"))
combined_df$channel <- sapply(strsplit(combined_df$feature, "_"), `[`, 1)

head(combined_df)

,feature,p_value,ks_stat,p_value_fdr_corrected,neg_log10_p_value,signature,channel,data_type
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<fct>
1,Cytoplasm_AreaShape_BoundingBoxMinimum_X,1.281552e-04,0.06355500,1.495524e-04,3.8252067,on,Cytoplasm,Non-shuffled
2,Cytoplasm_AreaShape_Compactness,7.846663e-01,0.01883572,7.894728e-01,0.1026628,off,Cytoplasm,Non-shuffled
3,Cytoplasm_AreaShape_Eccentricity,3.889191e-47,0.21161789,9.903871e-47,46.0041950,on,Cytoplasm,Non-shuffled
4,Cytoplasm_AreaShape_Extent,1.873207e-21,0.14228771,3.139533e-21,20.5031349,on,Cytoplasm,Non-shuffled
5,Cytoplasm_AreaShape_FormFactor,7.846663e-01,0.01883572,7.894728e-01,0.1026628,off,Cytoplasm,Non-shuffled
6,Cytoplasm_AreaShape_MajorAxisLength,7.015711e-42,0.19945032,1.652087e-41,40.7819672,on,Cytoplasm,Non-shuffled


## Visualizing Feature Significance

We generate two side-by-side scatter plots to visualize the significance and effect size of differences between failing and healthy cells:

- **X-axis:** KS statistic (effect size) – quantifies the magnitude of difference between distributions.
- **Y-axis:** -log10(FDR-corrected p-value) – quantifies the statistical significance of that difference.
- **Significance Threshold:** A horizontal dashed line at `-log10(0.05)` marks the boundary. Features above this line are statistically significant.

**Panel 1 - Feature significance by channel:**
Colors each feature according to its imaging channel (e.g., Cells, Cytoplasm, Nuclei). This helps identify if specific compartments or stains are driving the observed phenotypic differences.

**Panel 2 - Feature significance by signature type:**
Colors features based on whether they are classified as an **on-morphological signature** (passing the significance threshold) or an **off-morphological signature** (failing to pass). Off-morphological signatures effectively serve as a metric of control or baseline variation for high-content screening.

In [ ]:
# Configure plot dimensions: 2 columns (channel, signature) x 2 rows (non-shuffled, shuffled)
height <- 12
width <- 16
options(repr.plot.width = width, repr.plot.height = height)

# Generate a color palette for the different channels
n_channels <- length(unique(combined_df$channel))
dark2_palette <- brewer.pal(max(3, min(n_channels, 8)), "Dark2")

# Set a shared Y-axis limit across both datasets so rows are directly comparable
y_max <- max(combined_df$neg_log10_p_value[is.finite(combined_df$neg_log10_p_value)], na.rm = TRUE) * 1.1

# Panel 1: Feature significance colored by imaging channel, faceted by shuffled status
plot_channel <- ggplot(combined_df, aes(x = ks_stat, y = neg_log10_p_value, color = channel)) +
  geom_point(size = 2.5, alpha = 0.7) +
  geom_xsidedensity(aes(fill = channel), alpha = 0.4, color = NA, position = "identity") +
  geom_ysidedensity(aes(fill = channel), alpha = 0.4, color = NA, position = "identity") +
  scale_color_manual(values = dark2_palette) +
  scale_fill_manual(values = dark2_palette, guide = "none") +
  scale_y_continuous(limits = c(0, y_max), expand = expansion(mult = c(0.02, 0.05))) +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "gray40", linewidth = 0.5) +
  facet_grid(data_type ~ ., scales = "fixed") +
  labs(
    x = "KS statistic (effect size)",
    y = "-log10(FDR-corrected p-value)",
    title = "Feature significance by channel",
    color = "Channel"
  ) +
  theme_minimal(base_size = 26) +
  theme(
    plot.title = element_text(hjust = 0.5, face = "bold", size = 29),
    axis.title = element_text(size = 22, face = "bold"),
    axis.text = element_text(size = 20),
    strip.text = element_text(face = "bold", size = 22),
    strip.background = element_rect(fill = "gray92", color = NA),
    legend.position = "right",
    legend.title = element_text(face = "bold", size = 22),
    legend.text = element_text(size = 18),
    panel.grid.minor = element_blank(),
    plot.margin = margin(20, 10, 20, 20),
    ggside.panel.scale = 0.2
  )

# Panel 2: Feature significance colored by signature classification, faceted by shuffled status
plot_significant <- ggplot(combined_df, aes(x = ks_stat, y = neg_log10_p_value, color = signature)) +
  geom_point(size = 2.5, alpha = 0.7) +
  geom_xsidedensity(aes(fill = signature), alpha = 0.4, color = NA, position = "identity") +
  geom_ysidedensity(aes(fill = signature), alpha = 0.4, color = NA, position = "identity") +
  scale_color_manual(
    values = c("off" = "gray60", "on" = "#E41A1C"),
    labels = c("off" = "off-morphological", "on" = "on-morphological")
  ) +
  scale_fill_manual(
    values = c("off" = "gray60", "on" = "#E41A1C"),
    guide = "none"
  ) +
  scale_y_continuous(limits = c(0, y_max), expand = expansion(mult = c(0.02, 0.05))) +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "gray40", linewidth = 0.5) +
  facet_grid(data_type ~ ., scales = "fixed") +
  labs(
    x = "KS statistic (effect size)",
    y = "-log10(FDR-corrected p-value)",
    title = "Feature significance by signature",
    color = "Signature"
  ) +
  theme_minimal(base_size = 26) +
  theme(
    plot.title = element_text(hjust = 0.5, face = "bold", size = 29),
    axis.title = element_text(size = 22, face = "bold"),
    axis.text = element_text(size = 20),
    strip.text = element_text(face = "bold", size = 22),
    strip.background = element_rect(fill = "gray92", color = NA),
    legend.position = "right",
    legend.title = element_text(face = "bold", size = 22),
    legend.text = element_text(size = 18),
    panel.grid.minor = element_blank(),
    plot.margin = margin(20, 20, 20, 10),
    ggside.panel.scale = 0.2
  )

# Combine side-by-side: each panel has its own facet rows (non-shuffled top, shuffled bottom)
combined_plot <- plot_channel + plot_significant

# Save the combined visualization
output_png_path <- file.path(sig_plot_output_dir, "cfret_signature_significance_plots.png")
ggsave(output_png_path, combined_plot, width = width, height = height, dpi = 300, bg = "white")

combined_plot

These twin scatter plots illustrate the effect size (KS statistic) and statistical significance (-log10 FDR-corrected p-value) of differences between failing and healthy CFReT cardiac fibroblasts for various morphological features. 

- **Left Panel:** Features are colored by their respective cellular compartment or imaging channel (e.g., Cells, Cytoplasm, Nuclei).
- **Right Panel:** Features are colored by their classification into "on-morphological signatures" (red) or "off-morphological signatures" (grey) based on the FDR significance threshold (dashed horizontal line).

**Interpretation:**
- Features plotted higher and further to the right represent highly significant, large-magnitude morphological differences between healthy and failing cells.
- The separation in the right panel confirms a clear boundary between the active, discriminative "on-morphological signatures" and the unperturbed "off-morphological signatures."
- The "off-morphological signatures" remain below the significance threshold, highlighting their potential utility as stable background controls for high-content screening metrics.